# 01 - Preprocessing: Online Shoppers Purchasing Intention Dataset

**Sumber dataset:** [UCI Machine Learning Repository - Online Shoppers Purchasing Intention Dataset](https://archive.ics.uci.edu/dataset/468/online%20shoppers%20purchasing%20intention%20dataset)
(diunggah ulang sebagai Kaggle Dataset publik: `dimaspashaakrilian/online-shoppers-purchasing-intention-dataset`)

**Tujuan notebook:** membersihkan data, melakukan eksplorasi awal (EDA), encoding fitur kategorikal, dan membagi data menjadi train/test agar siap dipakai oleh notebook `02-modeling` untuk membandingkan beberapa model klasifikasi ringan (Logistic Regression, Decision Tree, Naive Bayes, KNN, Random Forest).

**Catatan environment:** notebook ini didesain untuk berjalan di **CPU** (Kaggle Notebook, accelerator = *None*, bukan GPU T4) karena semua model yang dibandingkan adalah model klasik yang ringan secara komputasi.

In [ ]:
import os
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


## 0. Memuat Data

Mencoba beberapa lokasi umum: input Kaggle Dataset, lalu fallback ke file lokal.

In [ ]:
import glob

FILENAME = "online_shoppers_intention.csv"

candidate_paths = [
    f"/kaggle/input/online-shoppers-purchasing-intention-dataset/{FILENAME}",
    f"../input/online-shoppers-purchasing-intention-dataset/{FILENAME}",
    FILENAME,
    f"../online+shoppers+purchasing+intention+dataset/{FILENAME}",
]

data_path = next((p for p in candidate_paths if os.path.exists(p)), None)

if data_path is None:
    # Fallback: cari di mana pun di bawah /kaggle/input (struktur mount Kaggle
    # bisa berbeda-beda, mis. /kaggle/input/datasets/<owner>/<slug>/...)
    matches = glob.glob(f"/kaggle/input/**/{FILENAME}", recursive=True)
    if matches:
        data_path = matches[0]

if data_path is None:
    raise FileNotFoundError(
        "Dataset tidak ditemukan. Pastikan Kaggle Dataset "
        "'dimaspashaakrilian/online-shoppers-purchasing-intention-dataset' sudah ditambahkan "
        "sebagai data source pada notebook ini (Add Data)."
    )

print("Membaca dataset dari:", data_path)
df = pd.read_csv(data_path)
df.shape


## 1. Eksplorasi Data Awal (EDA)

In [ ]:
df.head()


In [ ]:
df.info()


In [ ]:
# Cek missing value per kolom
missing = df.isnull().sum()
missing[missing > 0] if missing.sum() > 0 else print("Tidak ada missing value pada dataset ini.")


In [ ]:
# Cek baris duplikat
n_dupes = df.duplicated().sum()
print(f"Jumlah baris duplikat: {n_dupes} dari {len(df)} baris ({n_dupes/len(df):.2%})")


In [ ]:
# Distribusi target (Revenue) -> dataset ini imbalanced
target_dist = df["Revenue"].value_counts(normalize=True).rename("proportion").to_frame()
target_dist["count"] = df["Revenue"].value_counts()
target_dist


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(5, 4))
df["Revenue"].value_counts().plot(kind="bar", ax=ax, color=["#4C72B0", "#DD8452"])
ax.set_title("Distribusi Target: Revenue (Purchase Intention)")
ax.set_xlabel("Revenue")
ax.set_ylabel("Jumlah sesi")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
df.describe(include="all").T


**Catatan EDA:**
- Dataset berisi 12.330 baris sesi kunjungan e-commerce dengan 17 fitur + 1 target (`Revenue`).
- Tidak ada missing value.
- Terdapat sejumlah baris duplikat yang perlu dibuang.
- Target `Revenue` **imbalanced** (~85% False / ~15% True) sehingga metrik evaluasi di notebook modeling tidak boleh hanya mengandalkan *accuracy*, tapi juga precision, recall, F1, dan ROC-AUC.
- Fitur kategorikal nominal: `Month`, `VisitorType`, `Weekend`. Fitur `OperatingSystems`, `Browser`, `Region`, `TrafficType` sebenarnya berupa kode kategori (bukan angka kontinu) sehingga tetap diperlakukan sebagai kategorikal saat encoding.

## 2. Data Cleaning

In [ ]:
before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
after = len(df)
print(f"Baris sebelum: {before}, sesudah drop_duplicates: {after}, dibuang: {before - after}")


## 3. Feature Engineering & Encoding

Langkah:
1. Ubah kolom boolean (`Weekend`, `Revenue`) menjadi 0/1.
2. One-hot encoding untuk semua fitur kategorikal nominal: `Month`, `VisitorType`, `OperatingSystems`, `Browser`, `Region`, `TrafficType`.
3. Fitur numerik kontinu (`Administrative`, `ProductRelated_Duration`, `BounceRates`, `ExitRates`, `PageValues`, dst.) dibiarkan apa adanya di sini — proses *scaling* dilakukan di notebook modeling di dalam `Pipeline` (fit hanya pada data train) agar tidak terjadi data leakage.

In [ ]:
df["Weekend"] = df["Weekend"].astype(int)
df["Revenue"] = df["Revenue"].astype(int)

categorical_nominal = ["Month", "VisitorType", "OperatingSystems", "Browser", "Region", "TrafficType"]

df_encoded = pd.get_dummies(df, columns=categorical_nominal, drop_first=True)

# pastikan semua kolom hasil one-hot bertipe numerik (0/1) bukan bool
bool_cols = df_encoded.select_dtypes(include="bool").columns
df_encoded[bool_cols] = df_encoded[bool_cols].astype(int)

print("Shape sebelum encoding:", df.shape)
print("Shape sesudah encoding:", df_encoded.shape)
df_encoded.head()


In [ ]:
df_encoded.dtypes.value_counts()


## 4. Train-Test Split

Split stratified 80/20 berdasarkan target `Revenue` supaya proporsi kelas minoritas tetap terjaga di kedua subset.

In [ ]:
from sklearn.model_selection import train_test_split

X = df_encoded.drop(columns=["Revenue"])
y = df_encoded["Revenue"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print("X_train:", X_train.shape, "| X_test:", X_test.shape)
print("\nProporsi target - train:")
print(y_train.value_counts(normalize=True))
print("\nProporsi target - test:")
print(y_test.value_counts(normalize=True))


## 5. Simpan Hasil Preprocessing

Disimpan ke `/kaggle/working/` sebagai output notebook ini, supaya bisa dipakai sebagai *data source* oleh notebook `02-modeling` (via fitur "Add utility script/output" atau kernel-output-source di Kaggle).

In [ ]:
out_dir = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."

train_df = X_train.copy()
train_df["Revenue"] = y_train.values
test_df = X_test.copy()
test_df["Revenue"] = y_test.values

train_path = os.path.join(out_dir, "train.csv")
test_path = os.path.join(out_dir, "test.csv")

train_df.to_csv(train_path, index=False)
test_df.to_csv(test_path, index=False)

print("Tersimpan:", train_path, train_df.shape)
print("Tersimpan:", test_path, test_df.shape)


## Ringkasan

| Tahap | Hasil |
|---|---|
| Data mentah | 12.330 baris x 18 kolom |
| Setelah drop duplicate | lihat output di atas |
| Setelah one-hot encoding | lihat output di atas |
| Train set | 80%, stratified |
| Test set | 20%, stratified |

Lanjut ke notebook **`02-modeling`** untuk melatih dan membandingkan 5 model klasifikasi murah secara komputasi: **Logistic Regression, Decision Tree, Naive Bayes (Gaussian), K-Nearest Neighbors, dan Random Forest** — semuanya dijalankan di **CPU**, tanpa GPU.